# PennyLane Hamiltonian simulation

Trotterize transverse-field Ising dynamics and compare an expectation-value trajectory.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [1]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    total_variation_distance,
)

In [2]:
times = np.linspace(0.0, 1.2, 13)

def make_qnode(device):
    @qml.qnode(device)
    def evolution(time_value):
        qml.PauliX(0)
        steps = 6
        dt = time_value / steps
        for _ in range(steps):
            qml.IsingZZ(1.1 * dt, wires=[0, 1])
            qml.IsingZZ(1.1 * dt, wires=[1, 2])
            for wire in range(3):
                qml.RX(0.7 * dt, wires=wire)
        return qml.expval(qml.Z(0))
    return evolution

reference_qnode = make_qnode(qml.device("default.qubit", wires=3))
reference, reference_ms, _ = benchmark(lambda: np.asarray([reference_qnode(value) for value in times]))
mettleq_device = MettleQDevice(wires=3, method="statevector", device="cpu")
mettleq_qnode = make_qnode(mettleq_device)
candidate, mettleq_ms, _ = benchmark(lambda: np.asarray([mettleq_qnode(value) for value in times]))
error = max_abs_error(reference, candidate)
method, device = pennylane_selection(mettleq_device)
tutorial_result = emit_result(
    notebook="pennylane/11_hamiltonian_simulation.ipynb",
    framework="pennylane",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="observable trajectory atol=3e-6",
    passed=error <= 3e-6,
    exact_match=bool(np.array_equal(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"max_observable_error": error, "times": times, "reference": reference, "mettleq": candidate},
)

TUTORIAL_RESULT::{"check": "observable trajectory atol=3e-6", "exact_match": false, "framework": "pennylane", "machine": "arm64", "metrics": {"max_observable_error": 8.914426445905121e-07, "mettleq": [-1.0, -0.9975538849830627, -0.9902538061141968, -0.9782240390777588, -0.9616617560386658, -0.9408407211303711, -0.9161059260368347, -0.8878591060638428, -0.8565614819526672, -0.82271409034729, -0.7868609428405762, -0.7495627999305725, -0.7114009857177734], "reference": [-1.0, -0.9975533999646711, -0.9902542917948216, -0.9782239468098266, -0.9616618196615209, -0.9408416125730157, -0.916105884970557, -0.8878593249231737, -0.8565608274635526, -0.8227145502198749, -0.7868601382708603, -0.7495623272636099, -0.7114001462281644], "times": [0.0, 0.09999999999999999, 0.19999999999999998, 0.3, 0.39999999999999997, 0.49999999999999994, 0.6, 0.7, 0.7999999999999999, 0.8999999999999999, 0.9999999999999999, 1.0999999999999999, 1.2]}, "mettleq_median_ms": 21.754750021500513, "notebook": "pennylane/11_ha